In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_sl.order")

In [0]:
vehicle_df = order_df.withColumn("vehicleno",lit("NA"))

In [0]:
vehicle_df = vehicle_df.select("vehicleno","carriername","carriercity","carrierstate")\
    .withColumn("ingestion_time", current_timestamp()).distinct()

In [0]:
vehicle_df.createOrReplaceTempView("temp_vehicle")

In [0]:
%sql
merge into dev.tracking_gd.dimvehicle tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_vehicle 
) src 
on tgt.carrriername = src.carriername
  and tgt.carriercity = src.carriercity
  and tgt.carrierstate = src.carrierstate
when not matched then
  insert (vehicleno, carrriername, carriercity, carrierstate, ingestion_time)
  Values(src.vehicleno,src.carriername,src.carriercity,src.carrierstate,src.ingestion_time)